# 05 — Clasificación y confirmación de antimicrobianos

Aplica reglas versionadas a `prescriptions`, audita exclusiones y enlaza `emar` mediante `pharmacy_id`. Compara sospecha de infección basada en prescripción con una sensibilidad que exige administración documentada. Las reglas actuales cubren los nombres observados en el demo y deben revisarse clínicamente antes del análisis completo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path: sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from mimic_sepsis.antimicrobials import classify_prescriptions, confirm_administrations, load_antimicrobial_rules
from mimic_sepsis.infection import pair_antibiotics_and_cultures
DATA_DIR 
= PROJECT_ROOT / 'data' / 'mimic-iv-demo' / '2.2' / 'hosp'

In [ ]:
rules = load_antimicrobial_rules(PROJECT_ROOT / 'config' / 'antimicrobial_rules.csv')
prescriptions = pd.read_csv(DATA_DIR / 'prescriptions.csv.gz', low_memory=False)
emar = pd.read_csv(DATA_DIR / 'emar.csv.gz', low_memory=False)
microbiology = pd.read_csv(DATA_DIR / 'microbiologyevents.csv.gz', low_memory=False)
classified = classify_prescriptions(prescriptions, rules)
print(f'{len(rules)} reglas aplicadas a {len(classified)} prescripciones')

## Auditoría de clasificación

In [ ]:
classification_audit = classified.groupby('classification_reason').size().rename('prescriptions').reset_index().sort_values('prescriptions', ascending=False)
classification
_audit

In [ ]:
included_inventory = (classified.loc[classified.is_antimicrobial].groupby(['matched_pattern','antimicrobial_group','route'], dropna=False).size().rename('prescriptions').reset_index().sort_values('prescriptions', ascending=False))
included
_inventory

## Confirmación mediante EMAR

Se considera evidencia inicial `Administered`, `Delayed Administered`, `Started`, `Restarted` y variantes que contienen esos términos. `Not Given`, `Hold Dose`, `Stopped` y `Confirmed` no cuentan como administración.

In [ ]:
confirmed = confirm_administrations(classified, emar)
administration
_summary = pd.DataFrame({'metric':['prescripciones antimicrobianas','con administración EMAR','sin administración EMAR'],'count':[len(confirmed), confirmed.administration_time.notna().sum(), confirmed.administration_time.isna().sum()]})
administration_summary

## Cultivos deduplicados y pares temporales

In [ ]:
cultures = (microbiology.assign(culture_time=lambda x: pd.to_datetime(x.charttime, errors='coerce').fillna(pd.to_datetime(x.chartdate, errors='coerce'))).sort_values(['subject_id','hadm_id','micro_specimen_id','culture_time']).drop_duplicates(['subject_id','hadm_id','micro_specimen_id']).rename(columns={'micro_specimen_id':'culture_id'}))
culture_events = cultures[['subject_id','hadm_id','culture_id','culture_time']]
blood_events = cultures.loc[cultures.spec_type_desc.fillna('').str.contains('BLOOD', case=False), ['subject_id','hadm_id','culture_id','culture_time']]
rx_
events = classified.loc[classified.is_antimicrobial, ['subject_id','hadm_id','pharmacy_id','prescription_time']].rename(columns={'pharmacy_id':'antibiotic_id','prescription_time':'antibiotic_time'})
admin_events = confirmed.dropna(subset=['administration_time'])[['subject_id','hadm_id','pharmacy_id','administration_time']].rename(columns={'pharmacy_id':'antibiotic_id','administration_time':'antibiotic_time'})

In [ ]:
comparisons = []
for exposure_name, exposure in [('prescription', rx_events), ('emar_administration', admin_events)]:
    for culture_name, culture_set in [('all_specimens', culture_events), ('blood_only', blood_events)]:
        pairs = pair_antibiotics_and_cultures(exposure, culture_set)
        comparisons.append({'exposure':exposure_name,'cultures':culture_name,'pairs':len(pairs),'admissions':pairs.hadm_id.nunique()})
pd.DataFrame(comparisons)

## Estado metodológico

La definición compatible con `mimic-code` basada en prescripción se conservará como comparador reproducible. La variante con administración EMAR reduce la exposición no administrada, pero puede perder eventos por enlaces incompletos y cambios de flujo de trabajo. La definición primaria se congelará después de revisar cobertura por fármaco, vía, UCI y versión completa; sangre será primaria y todos los especímenes una sensibilidad.